# Engine comparison: free fit vs Engine A vs Engine B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/calib_anchor_compare.ipynb)

Loads the full-game registration chain `.npz` + the session's exported `keypoints.parquet`, runs `compare_engines`, prints the coverage/fold/accuracy table, plots per-frame median reprojection drift, times each engine, and renders overlays for visual assessment. CPU-only (no GPU).

In [ ]:
# --- Colab setup: install the package + stage data from Drive ---
# (skip the Drive block if running locally and edit CHAIN/KEYPOINTS/VIDEO to local paths)
!pip install -q "soccer-vision @ git+https://github.com/PatrickJReed/soccer-vision.git#subdirectory=packages/soccer-vision"

from pathlib import Path

# Edit to your filenames under Google Drive MyDrive/soccer-vision/
DRIVE_CHAIN = "training_chain.npz"             # the labeler chain .npz (rename the <hash>.npz)
DRIVE_KEYPOINTS = "training_keypoints.parquet" # exported keypoints.parquet
DRIVE_VIDEO = "training.mp4"                    # optional: full-game video (overlays cell only)

CHAIN = Path("/content/chain.npz")
KEYPOINTS = Path("/content/keypoints.parquet")
VIDEO = Path("/content/game.mp4")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/soccer-vision")
    !cp "{DRIVE/DRIVE_CHAIN}" "{CHAIN}"
    !cp "{DRIVE/DRIVE_KEYPOINTS}" "{KEYPOINTS}"
    if (DRIVE / DRIVE_VIDEO).exists():
        !cp "{DRIVE/DRIVE_VIDEO}" "{VIDEO}"
except ModuleNotFoundError:
    pass  # not on Colab -> set CHAIN/KEYPOINTS/VIDEO to local paths above

assert CHAIN.exists() and KEYPOINTS.exists(), \
    "put the chain .npz + keypoints.parquet in Drive/soccer-vision/ (or set local paths)"
print("staged:", CHAIN, "|", KEYPOINTS, "| video present:", VIDEO.exists())

In [ ]:
# --- load the chain + clicks ---
from soccer_vision.labeler.chain import load_chain
from soccer_vision.labeler.state import clicks_from_keypoints_parquet

chain_result = load_chain(CHAIN)
assert chain_result is not None, f"chain not found: {CHAIN}"
interframe, n_frames, size = chain_result
print(f"chain: {len(interframe)} inter-frame transforms, {n_frames} frames, size={size}")

clicks = clicks_from_keypoints_parquet(KEYPOINTS, size)
print(f"clicks: {len(clicks)} observations across {len({c.frame for c in clicks})} frames")

In [ ]:
# --- coverage / folds / accuracy ---
from soccer_vision.pitch.calib_compare import compare_engines

result = compare_engines(clicks, interframe, n_frames, size)

print(f"{'engine':<12} {'covered':>8} {'cov%':>7} {'folded':>8} {'median_ft':>10} {'p90_ft':>8}")
print("-" * 57)
for name, m in result.items():
    print(
        f"{name:<12} {m.n_covered:>8} {m.coverage_fraction*100:>6.1f}%"
        f" {m.n_folded:>8} {m.median_accuracy_ft:>10.2f} {m.p90_accuracy_ft:>8.2f}"
    )

In [ ]:
# --- drift: per-frame median reprojection error vs frame number ---
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
colors = {"free_fit": "steelblue", "engine_a": "darkorange", "engine_b": "forestgreen"}
for name, m in result.items():
    if m.per_frame_median_ft:
        frames = sorted(m.per_frame_median_ft)
        vals = [m.per_frame_median_ft[f] for f in frames]
        ax.scatter(frames, vals, s=12, alpha=0.7, label=name, color=colors[name])
ax.set_xlabel("frame")
ax.set_ylabel("median reprojection error (ft)")
ax.set_title("Per-frame accuracy drift by engine")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- runtime: time each engine independently ---
import time
from soccer_vision.pitch.manual_anchor import build_segments, cumulative_transforms, fit_frame_homographies
from soccer_vision.pitch.calib_anchor import (
    calibrate_clicked_frames, poses_by_click_propagation, poses_by_pose_propagation,
)
from soccer_vision.pitch.landmarks import PITCH_LANDMARKS

segment_of = build_segments(interframe, n_frames)
transforms = cumulative_transforms(interframe, segment_of)
k, clicked_poses = calibrate_clicked_frames(clicks, size)

t0 = time.perf_counter()
fit_frame_homographies(clicks, transforms, segment_of, PITCH_LANDMARKS, window=360)
t_free = time.perf_counter() - t0

t0 = time.perf_counter()
poses_by_click_propagation(clicks, transforms, segment_of, k, size, window=360)
t_a = time.perf_counter() - t0

t0 = time.perf_counter()
poses_by_pose_propagation(transforms, segment_of, k, clicked_poses, size)
t_b = time.perf_counter() - t0

print(f"free_fit : {t_free*1000:.0f} ms")
print(f"engine_a : {t_a*1000:.0f} ms")
print(f"engine_b : {t_b*1000:.0f} ms")

In [ ]:
# --- held-out reference (optional): leave-one-out accuracy on the clicked frames ---
from soccer_vision.calib.validate import leave_one_out_feet
import numpy as np

w, h = size
obs = {}
for c in clicks:
    obs.setdefault(c.frame, []).append((c.kp_idx, c.x * w, c.y * h))

loo = leave_one_out_feet(obs, k, size, min_other=4)
flat_loo = [v for vals in loo.values() for v in vals]
if flat_loo:
    print(f"Leave-one-out (calib model): median {np.median(flat_loo):.2f} ft  "
          f"90th {np.percentile(flat_loo, 90):.2f} ft  ({len(flat_loo)} obs)")
else:
    print("leave_one_out_feet: no observations (need >=5 per frame)")

In [ ]:
# --- overlays: render free-fit / A / B pitch on a few reference frames ---
# Patrick assesses; Claude renders only.
import cv2
import numpy as np
from soccer_vision.viz.pitch_overlay import draw_reprojected_pitch
from soccer_vision.pitch.manual_anchor import fit_frame_homographies
from soccer_vision.pitch.calib_anchor import (
    poses_by_click_propagation, poses_by_pose_propagation, frame_homography,
)
from soccer_vision.pitch.landmarks import PITCH_LANDMARKS

w_px, h_px = size
SAMPLE_FRAMES = sorted({c.frame for c in clicks})[:6] if VIDEO.exists() else []  # first 6 clicked frames
if not SAMPLE_FRAMES:
    print("no video staged (set DRIVE_VIDEO in the setup cell) -- skipping overlays")

free_fits = fit_frame_homographies(clicks, transforms, segment_of, PITCH_LANDMARKS, window=360)
a_poses = poses_by_click_propagation(clicks, transforms, segment_of, k, size, window=360)
b_poses = poses_by_pose_propagation(transforms, segment_of, k, clicked_poses, size)

# All three as FULL-PIXEL image->pitch[0,1]. FrameFit.H is normalized image->pitch,
# so denormalize it (H_px = H_norm @ diag(1/W,1/H,1)); calib homographies are already full-pixel.
_denorm = np.diag([1.0 / w_px, 1.0 / h_px, 1.0])


def _h_img2pitch(label, f):
    if label == "free":
        return free_fits[f].H @ _denorm if f in free_fits else None
    if label == "A":
        return frame_homography(k, a_poses[f].rvec, a_poses[f].tvec) if f in a_poses else None
    return frame_homography(k, b_poses[f].rvec, b_poses[f].tvec) if f in b_poses else None


def _overlay(frame, h):
    # project the 21 canonical landmarks to pixels via inv(H) (= pitch[0,1]->px) and draw
    if h is None:
        return frame
    h_inv = np.linalg.inv(np.asarray(h, dtype=np.float64))
    canon = np.column_stack([PITCH_LANDMARKS, np.ones(len(PITCH_LANDMARKS))])
    proj = canon @ h_inv.T
    px = (proj[:, :2] / proj[:, 2:3]).astype(np.float64)
    vis, _ = draw_reprojected_pitch(frame, px, np.arange(len(PITCH_LANDMARKS)))
    return vis


cap = cv2.VideoCapture(str(VIDEO))
tiles = []
for f in SAMPLE_FRAMES:
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, img = cap.read()
    if not ok:
        continue
    row_imgs = []
    for label in ("free", "A", "B"):
        vis = _overlay(img.copy(), _h_img2pitch(label, f))
        cv2.putText(vis, label, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 2)
        row_imgs.append(cv2.resize(vis, (640, 360)))
    tiles.append(np.hstack(row_imgs))
cap.release()

if tiles:
    out_path = "/tmp/calib_compare_overlay.jpg"
    cv2.imwrite(out_path, np.vstack(tiles))
    print(f"wrote {out_path}  ({len(tiles)} rows = frames x 3 engines [free | A | B])")
else:
    print("no frames rendered -- check VIDEO path")
